> **Historical predecessor.** Earlier transaction/sentiment experiments; not the current study.
> Open [Current_Study_Reproduction.ipynb](notebooks/Current_Study_Reproduction.ipynb) for the current manuscript workflow. Original code and outputs below are preserved for provenance.

<a href="https://colab.research.google.com/github/aaronab810/Dubai-Real-Estate-NLP/blob/main/dfm_transaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

In [ ]:
#test upload for github - praj

In [ ]:
dld_path = '/content/drive/MyDrive/Dubai_Real_Estate_Data/transactions.csv'
dfm_path = '/content/drive/MyDrive/Dubai_Real_Estate_Data/dfm.xlsx'

# 3. Load the raw data into Pandas DataFrames
dld_df = pd.read_csv(dld_path)
dfm_df = pd.read_excel(dfm_path)

# 4. Print the exact column headers so we can plan the merge
print(" DLD Columns ---")
print(dld_df.columns.tolist())
print("\n--- DFM Columns ---")
print(dfm_df.columns.tolist())

In [ ]:
import pandas as pd

# 1. Fix the DFM typo
dfm_df.rename(columns={'Chage %': 'Change %'}, inplace=True)

# 2. Convert both date columns to actual Pandas datetime objects
dld_df['INSTANCE_DATE'] = pd.to_datetime(dld_df['INSTANCE_DATE'], errors='coerce')
dfm_df['Date'] = pd.to_datetime(dfm_df['Date'], errors='coerce')

# 3. Filter DLD strictly for NLP correlation (ADDED ACTUAL_AREA!)
dld_features = ['INSTANCE_DATE', 'TRANS_VALUE', 'AREA_EN', 'PROCEDURE_EN', 'ACTUAL_AREA']
dld_clean = dld_df[dld_features].copy()

# 4. Filter for actual Sales using string matching to avoid 0 rows
dld_clean = dld_clean[dld_clean['PROCEDURE_EN'].str.contains('Sale', case=False, na=False)]

print(f"Remaining DLD rows after Sale filter: {len(dld_clean)}")

# 5. Drop the PROCEDURE_EN column now that we're done filtering
dld_clean.drop(columns=['PROCEDURE_EN'], inplace=True)

# 6. Calculate Price per SqFt
# (Adding a small check to drop any rows where area is 0 to avoid math errors)
dld_clean = dld_clean[dld_clean['ACTUAL_AREA'] > 0]
dld_clean['PRICE_SQFT'] = dld_clean['TRANS_VALUE'] / dld_clean['ACTUAL_AREA']

# 7. Group by District and Week
weekly = (dld_clean.groupby([
            'AREA_EN',
            pd.Grouper(key='INSTANCE_DATE', freq='W')
          ])
          .agg(
              transaction_count=('TRANS_VALUE', 'count'),
              median_price_sqft=('PRICE_SQFT', 'median'),
              total_value=('TRANS_VALUE', 'sum')
          )
          .round(2)
          .reset_index()
)

# 8. Sort chronologically per district
weekly = weekly.sort_values(['AREA_EN', 'INSTANCE_DATE'])

print("\nWeekly district aggregation complete.")
print(weekly.head())

In [ ]:
# 1. Calculate the 12-week (3-month) rolling average of the median price per sqft
# We use transform to keep the index aligned, and min_periods=1 so early weeks don't become NaN
weekly['rolling_3m_price_sqft'] = (
    weekly.groupby('AREA_EN')['median_price_sqft']
    .transform(lambda x: x.rolling(window=12, min_periods=1).mean())
    .round(2)
)

# 2. Aggregate DFM data to weekly (taking the last trading day's close for that week)
dfm_weekly = dfm_df.groupby(pd.Grouper(key='Date', freq='W')).agg(
    DFM_Close=('DFM Index', 'last')
).reset_index()

# 3. Rename the DFM date column so it exactly matches the DLD date column
dfm_weekly.rename(columns={'Date': 'INSTANCE_DATE'}, inplace=True)

# 4. Merge the two datasets together using an inner join!
# This ensures we only keep weeks where we have both Real Estate data AND Stock Index data
master_dataset = pd.merge(weekly, dfm_weekly, on='INSTANCE_DATE', how='inner')

print("Target variable calculated and DFM merged successfully.")
print(master_dataset.head())

In [ ]:
# 1. Define the exact save path in your Google Drive folder
output_path = '/content/drive/MyDrive/Dubai_Real_Estate_Data/dfm_transaction.csv'

# 2. Export the dataframe to a CSV file
# (Setting index=False ensures Pandas doesn't add a weird unnamed number column to the start of your file)
master_dataset.to_csv(output_path, index=False)

print(f"Boom. Baseline locked and loaded.")
print(f"File saved permanently to: {output_path}")

In [ ]:
print("--- Missing Values Check ---")
print(master_dataset.isnull().sum())

print("\n--- Statistical Sanity Check ---")
# Formatting to avoid scientific notation
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print(master_dataset.describe())

In [ ]:
# Check the timeline for the district we looked at earlier
al_barari_test = master_dataset[master_dataset['AREA_EN'] == 'AL BARARI'].copy()

print("Checking Al Barari Timeline Sequence...")
print(al_barari_test[['INSTANCE_DATE', 'median_price_sqft', 'rolling_3m_price_sqft', 'DFM_Close']].head(10))

In [ ]:
import matplotlib.pyplot as plt

# Plotting the dual-axis chart for Al Barari
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot 1: Your Real Estate Target (Left Axis)
ax1.set_xlabel('Date')
ax1.set_ylabel('Rolling 3-Month Price / SqFt (AED)', color='tab:blue', fontweight='bold')
ax1.plot(al_barari_test['INSTANCE_DATE'], al_barari_test['rolling_3m_price_sqft'], color='tab:blue', label='Smoothed Property Price', linewidth=2)
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Plot 2: The DFM Index (Right Axis)
ax2 = ax1.twinx()
ax2.set_ylabel('DFM Index Close', color='tab:red', fontweight='bold')
ax2.plot(al_barari_test['INSTANCE_DATE'], al_barari_test['DFM_Close'], color='tab:red', linestyle='--', label='DFM Index')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Validation Plot: Al Barari Rolling Price vs DFM Index')
fig.tight_layout()
plt.show()

In [ ]:
# Check which districts have the most 'signal' (transactions)/ see
#which districts actually have enough data to be useful for your FinBERT model

district_stats = (master_dataset.groupby('AREA_EN')
                  .agg(
                      total_weeks=('INSTANCE_DATE', 'count'),
                      avg_weekly_trans=('transaction_count', 'mean'),
                      price_volatility=('median_price_sqft', 'std')
                  )
                  .sort_values('avg_weekly_trans', ascending=False)
)

print("Top 10 Districts by Activity:")
print(district_stats.head(10))

be 100% sure the rolling average is working correctly for everyone, let’s plot the Top 3 most active districts against the DFM index. If the lines look independent of each other but all follow the DFM timeline, the merge is perfect.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Get the names of the top 3 districts
top_districts = district_stats.index[:3].tolist()

plt.figure(figsize=(14, 7))

# Plot the 3-month rolling average for each top district
for district in top_districts:
    subset = master_dataset[master_dataset['AREA_EN'] == district]
    plt.plot(subset['INSTANCE_DATE'], subset['rolling_3m_price_sqft'], label=f'Price: {district}')

plt.title('Rolling 3-Month Price Trends: Top 3 Districts')
plt.ylabel('AED per SqFt')
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.show()

need to confirm that your .transform() function stayed within its "lane." If the rolling average for "District A" accidentally included prices from "District B," your AI will learn absolute gibberish.

In [ ]:
# Pick two districts with vastly different price points
palm = master_dataset[master_dataset['AREA_EN'].str.contains('PALM', case=False, na=False)]['rolling_3m_price_sqft'].mean()
discovery = master_dataset[master_dataset['AREA_EN'].str.contains('DISCOVERY', case=False, na=False)]['rolling_3m_price_sqft'].mean()

print(f"Average Rolling Price (Palm): {palm:.2f} AED")
print(f"Average Rolling Price (Discovery Gardens): {discovery:.2f} AED")

if abs(palm - discovery) > 500:
    print("\n PASS: Districts are mathematically isolated. No data leakage detected.")
else:
    print("\n WARNING: Prices look suspiciously similar. Double-check the groupby logic.")

In [ ]:
!pip install apify-client

In [ ]:
from apify_client import ApifyClient
import pandas as pd

# 1. Initialize the Client
client = ApifyClient('apify_api_BFDHLypo9WQnN3QwyUmbWlbdzNfevb2JYWsK')

# 2. Corrected Input Structure
# The actor expects 'startUrls' as a list of dictionaries with a 'url' key
# Use your actual district names as the search query
districts_to_track = "OR ".join(['"JVC"', '"Business Bay"', '"Dubai Marina"', '"JLT"'])

run_input = {
    "startUrls": [
        { "url": "https://www.facebook.com/groups/DubaiPropertyInvestors/" },
        { "url": "https://www.facebook.com/groups/2485536551677335/" }
    ],
    "maxPosts": 40,
    # This is an objective geographical filter
    "searchTerm": districts_to_track,
    "viewOption": "CHRONOLOGICAL"
}

# 3. Run the Actor
print("Starting the scrape... (Watch your Apify dashboard for live progress!)")
run = client.actor("apify/facebook-groups-scraper").call(run_input=run_input)

# 4. Fetch results from the dataset
dataset_items = client.dataset(run["defaultDatasetId"]).list_items().items

# 5. Convert to DataFrame
fb_sentiment_df = pd.DataFrame(dataset_items)

print(f"Scrape complete! Captured {len(fb_sentiment_df)} posts.")
fb_sentiment_df.head()

In [ ]:
# Objective mapping in Colab
def map_district(text):
    for district in ['Jumeirah Village Circle', 'Business Bay', 'Dubai Marina']:
        if district.lower() in text.lower() or (district == 'Jumeirah Village Circle' and 'jvc' in text.lower()):
            return district
    return "General"

clean_sentiment_df['assigned_district'] = clean_sentiment_df['text'].apply(map_district)

running on mock data to test the pipeline. it works.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
import pandas as pd

# Load the model and tokenizer
model_name = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Initialize the analyzer
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

def get_sentiment_score(text):
    if not text or pd.isna(text):
        return "Neutral"
    # FinBERT handles 512 tokens max
    return sentiment_analyzer(text[:512])[0]['label']

In [ ]:
# Mock data aligned with your top districts from image_993259.png
mock_data = {
    'text': [
        "JVC is showing great value this month, very happy with the maintenance here.",
        "Business Bay traffic is getting worse, might affect rental yields soon.",
        "Jumeirah Village Circle properties are still the best bang for buck in Dubai.",
        "Service charges in Business Bay are too high compared to the actual facilities.",
        "Dubai Marina remains the gold standard for high-end luxury living."
    ],
    'timestamp': ['2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24'],
    'district': ['JUMEIRAH VILLAGE CIRCLE', 'BUSINESS BAY', 'JUMEIRAH VILLAGE CIRCLE', 'BUSINESS BAY', 'DUBAI MARINA']
}

fb_sentiment_df = pd.DataFrame(mock_data)
fb_sentiment_df['sentiment'] = fb_sentiment_df['text'].apply(get_sentiment_score)

In [ ]:
# Convert dates to datetime objects for matching
fb_sentiment_df['timestamp'] = pd.to_datetime(fb_sentiment_df['timestamp'])
master_dataset['INSTANCE_DATE'] = pd.to_datetime(master_dataset['INSTANCE_DATE'])

# Pivot sentiment to get a "Weekly Sentiment Score" per district
# For MS1, we'll map Negative = -1, Neutral = 0, Positive = 1
sentiment_map = {'Negative': -1, 'Neutral': 0, 'Positive': 1}
fb_sentiment_df['sentiment_val'] = fb_sentiment_df['sentiment'].map(sentiment_map)

weekly_sentiment = fb_sentiment_df.groupby(['district', pd.Grouper(key='timestamp', freq='W-MON')])['sentiment_val'].mean().reset_index()

# Final Join: Match on District and Date
final_analysis_df = pd.merge(
    master_dataset,
    weekly_sentiment,
    left_on=['AREA_EN', 'INSTANCE_DATE'],
    right_on=['district', 'timestamp'],
    how='left'
)

In [ ]:
import matplotlib.pyplot as plt

# Filter for a specific district to keep the plot clean
target_district = 'JUMEIRAH VILLAGE CIRCLE'
plot_df = final_analysis_df[final_analysis_df['AREA_EN'] == target_district]

fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot 1: Median Price (Left Axis)
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Median Price per SqFt', color=color)
ax1.plot(plot_df['INSTANCE_DATE'], plot_df['rolling_3m_price_sqft'], color=color, linewidth=3, label='Price Trend')
ax1.tick_params(axis='y', labelcolor=color)

# Create a second axis for Sentiment
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Sentiment Score (-1 to 1)', color=color)
ax2.bar(plot_df['INSTANCE_DATE'], plot_df['sentiment_val'], color=color, alpha=0.3, width=5, label='Sentiment')
ax2.set_ylim(-1.5, 1.5)
ax2.axhline(0, color='black', lw=1, ls='--')

plt.title(f'Price vs. Sentiment Alignment: {target_district}')
fig.tight_layout()
plt.show()

In [ ]:
# 1. Check for NaNs in the merged column
merged_rows = final_analysis_df['sentiment_val'].notna().sum()
print(f"Total Rows with valid sentiment: {merged_rows}")

# 2. Inspect the date formats
print(f"Transaction Date (first row): {master_dataset['INSTANCE_DATE'].iloc[0]}")
print(f"Sentiment Date (first row): {weekly_sentiment['timestamp'].iloc[0]}")

In [ ]:
# Force both to the start of the week (Monday)
master_dataset['INSTANCE_DATE'] = pd.to_datetime(master_dataset['INSTANCE_DATE']).dt.to_period('W-MON').dt.to_timestamp()
weekly_sentiment['timestamp'] = pd.to_datetime(weekly_sentiment['timestamp']).dt.to_period('W-MON').dt.to_timestamp()

# Re-run the merge
final_analysis_df = pd.merge(
    master_dataset,
    weekly_sentiment,
    left_on=['AREA_EN', 'INSTANCE_DATE'],
    right_on=['district', 'timestamp'],
    how='left'
).fillna({'sentiment_val': 0}) # Fill empty weeks with Neutral (0)

print("Merge re-aligned. Try the plot again!")

In [ ]:
import matplotlib.pyplot as plt

# Filter for a specific district to keep the plot clean
target_district = 'JUMEIRAH VILLAGE CIRCLE'
plot_df = final_analysis_df[final_analysis_df['AREA_EN'] == target_district]

fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot 1: Median Price (Left Axis)
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Median Price per SqFt', color=color)
ax1.plot(plot_df['INSTANCE_DATE'], plot_df['rolling_3m_price_sqft'], color=color, linewidth=3, label='Price Trend')
ax1.tick_params(axis='y', labelcolor=color)

# Create a second axis for Sentiment
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Sentiment Score (-1 to 1)', color=color)
ax2.bar(plot_df['INSTANCE_DATE'], plot_df['sentiment_val'], color=color, alpha=0.3, width=5, label='Sentiment')
ax2.set_ylim(-1.5, 1.5)
ax2.axhline(0, color='black', lw=1, ls='--')

plt.title(f'Price vs. Sentiment Alignment: {target_district}')
fig.tight_layout()
plt.show()